<a href="https://colab.research.google.com/github/mutagi/GenAIEngineering-Cohort2/blob/main/GPT_2_IndianChef.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial

Modified from [this notebook](https://colab.research.google.com/github/philschmid/fine-tune-GPT-2/blob/master/Fine_tune_a_non_English_GPT_2_Model_with_Huggingface.ipynb?utm_source=chatgpt.com#scrollTo=acdgnDUAgB1U)

In the tutorial, we are going to fine-tune a GPT-2 from the [Huggingface model hub](https://huggingface.co/models).

The idea is we use the recipe description to fine-tune our GPT-2 to let us write recipes we can cook.

I am using Google Colab with a GPU runtime for this tutorial. If you are not sure how to use a GPU Runtime take a look here.

## **What are we going to do:**

- load the dataset from kaggle
- prepare the dataset and build a ``TextDataset``
- load the pre-trained GPT-2 model and tokenizer
- initialize ``Trainer`` with ``TrainingArguments``
- train and save the model
- test the model

In [ ]:
!pip install transformers==4.2.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.2/184.2 kB 11.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 33.5 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [ ]:
!nvidia-smi

Sat Sep 20 06:08:35 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Load the dataset from Kaggle

As already mentioned in the introduction of the tutorial we use the "[German Recipes Dataset](https://www.kaggle.com/sterby/german-recipes-dataset)" dataset from Kaggle. The dataset consists of 5928 indian recipes with metadata crawled In this example, we only use the Instructions of the recipes. You can either download the dataset by using the "Download" Button and uploading it to our colab notebook.



In [ ]:
#upload files to your colab environment
from google.colab import files
uploaded = files.upload()

Saving archive.zip to archive.zip


After we uploaded the file with use `unzip` to extract the recipes.json.

In [ ]:
!unzip 'archive.zip'

Archive:  archive.zip
  inflating: Cleaned_Indian_Food_Dataset.csv  


# Prepare the dataset and build a ``TextDataset``

The next step is to extract the instructions from all recipes and build a `TextDataset`. The `TextDataset` is a custom implementation of the [Pytroch `Dataset` class](https://pytorch.org/tutorials/beginner/data_loading_tutorial.html#dataset-class) implemented by the transformers library. If you want to know more about Dataset in Pytroch you can check out this [youtube video](https://www.youtube.com/watch?v=PXOzkkB5eH0&ab_channel=PythonEngineer).

First, we are going to split the `recipes.json` into a `train` and `test` section and extract `Instructions` from the recipes and write them into a `train_dataset.txt` and `test_dataset.txt`

In [ ]:
import re
import json
from sklearn.model_selection import train_test_split
import pandas as pd

# Load the CSV file into a pandas DataFrame
df = pd.read_csv('Cleaned_Indian_Food_Dataset.csv')

# Combine the 'TranslatedIngredients' and 'TranslatedInstructions' columns
# Handle potential missing values by converting to string and filling NaN with empty string
df['CombinedRecipe'] = df['TranslatedRecipeName'].fillna('') + " : " + df['TranslatedIngredients'].fillna('').astype(str) + " " + df['TranslatedInstructions'].fillna('').astype(str)

def build_text_files(dataframe, dest_path):
    f = open(dest_path, 'w')
    data = ''
    for index, row in dataframe.iterrows():
        # Use the combined recipe column
        combined_text = str(row['CombinedRecipe']).strip()
        combined_text = re.sub(r"\s+", " ", combined_text) # Use + to handle multiple spaces
        data += combined_text + "\n"
    f.write(data)

# Split the dataframe into training and testing sets
train_df, test_df = train_test_split(df, test_size=0.15, random_state=42) # Added random_state for reproducibility

build_text_files(train_df,'train_dataset.txt')
build_text_files(test_df,'test_dataset.txt')

print("Train dataset length: "+str(len(train_df)))
print("Test dataset length: "+ str(len(test_df)))

Train dataset length: 5047
Test dataset length: 891


the next step is to download the tokenizer, which we use. We use the tokenizer from the `german-gpt2` model on [huggingface](https://huggingface.co/anonymous-german-nlp/german-gpt2).

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")

train_path = 'train_dataset.txt'
test_path = 'test_dataset.txt'

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
from transformers import TextDataset,DataCollatorForLanguageModeling

def load_dataset(train_path,test_path,tokenizer):
    train_dataset = TextDataset(
          tokenizer=tokenizer,
          file_path=train_path,
          block_size=128, # This is the maximum sequence length
          overwrite_cache=True # Add this to overwrite the cache
          )

    test_dataset = TextDataset(
          tokenizer=tokenizer,
          file_path=test_path,
          block_size=128, # This is the maximum sequence length
          overwrite_cache=True # Add this to overwrite the cache
          )

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer, mlm=False,
    )
    return train_dataset,test_dataset,data_collator

train_dataset,test_dataset,data_collator = load_dataset(train_path,test_path,tokenizer)

/usr/local/lib/python3.12/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (1991301 > 1024). Running this sequence through the model will result in indexing errors


# Initialize `Trainer` with `TrainingArguments` and GPT-2 model

The [Trainer](https://huggingface.co/transformers/main_classes/trainer.html#transformers.Trainer) class provides an API for feature-complete training. It is used in most of the [example scripts](https://huggingface.co/transformers/examples.html) from Huggingface. Before we can instantiate our `Trainer` we need to download our GPT-2 model and create a [TrainingArguments](https://huggingface.co/transformers/main_classes/trainer.html#transformers.TrainingArguments) to access all the points of customization during training. In the `TrainingArguments`, we can define the Hyperparameters we are going to use in the training process like our `learning_rate`, `num_train_epochs`, or  `per_device_train_batch_size`. A complete list can you find [here](https://huggingface.co/transformers/main_classes/trainer.html#trainingarguments).

In [ ]:
from transformers import Trainer, TrainingArguments,AutoModelWithLMHead

model = AutoModelWithLMHead.from_pretrained("openai-community/gpt2")


training_args = TrainingArguments(
    output_dir="./gpt2-indianchef", #The output directory
    overwrite_output_dir=True, #overwrite the content of the output directory
    num_train_epochs=3, # number of training epochs
    per_device_train_batch_size=32, # batch size for training
    per_device_eval_batch_size=64,  # batch size for evaluation
    eval_steps = 400, # Number of update steps between two evaluations.
    save_steps=800, # after # steps model is saved
    warmup_steps=500,# number of warmup steps for learning rate scheduler
    prediction_loss_only=True,
    )


trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2221: FutureWarning: The class `AutoModelWithLMHead` is deprecated and will be removed in a future version. Please use `AutoModelForCausalLM` for causal language models, `AutoModelForMaskedLM` for masked language models and `AutoModelForSeq2SeqLM` for encoder-decoder models.
  warnings.warn(


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

# Train and save the model

To train the model we can simply run `Trainer.train()`.

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kingsidharth (gs-test) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,2.812100
1000,2.240200


TrainOutput(global_step=1461, training_loss=2.396582967038843, metrics={'train_runtime': 1694.2, 'train_samples_per_second': 27.548, 'train_steps_per_second': 0.862, 'total_flos': 3048690106368000.0, 'train_loss': 2.396582967038843, 'epoch': 3.0})

After training is done you can save the model by calling `save_model()`. This will save the trained model to our `output_dir` from our `TrainingArguments`.

In [ ]:
trainer.save_model()

# Test the model

To test the model we are going to use another [highlight of the transformers library](https://huggingface.co/transformers/main_classes/pipelines.html?highlight=pipelines) called `pipeline`. [Pipelines](https://huggingface.co/transformers/main_classes/pipelines.html?highlight=pipelines) are objects that offer a simple API dedicated to several tasks, among others also `text-generation`

In [ ]:
from transformers import pipeline, AutoConfig

# Load the configuration from the saved model directory
config = AutoConfig.from_pretrained('./gpt2-indianchef')

chef = pipeline('text-generation',model='./gpt2-indianchef', tokenizer='openai-community/gpt2', config=config)

#result = chef('Zuerst Hähnchen')[0]['generated_text']

Device set to use cuda:0


In [ ]:
chef('To begin making the Masala Karela')

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'To begin making the Masala Karela Recipe, get all the ingredients ready. Heat a kadai with oil, add mustard seeds and allow it to crackle.Add curd, curry leaves, red chillies, chilli flakes and give it a stir.Add the mustard seeds and let it splutter. Add the bay leaf and allow it to crackle again. Add the turmeric powder and stir fry for a few seconds. Add the red chili powder and curry leaves and cook for another 10 minutes. Add in the chicken, stir and serve hot.Serve the Masala Karela Recipe along with steamed rice, Rajma Vada and steamed rice for a weeknight dinner.\nFried Pumpkin Bread Recipe : 1 cup All Purpose Flour (Maida),1/2 teaspoon Salt,1 teaspoon Black pepper powder,1/2 cup Whole Wheat Bread crumbs,1 Apple - cut into wedges,1 teaspoon Red Chilli flakes,1 cup Paneer (Homemade Cottage Cheese) - cut into wedges,1/2 teaspoon Cumin seeds (Jeera),2 Green Chillies - finely chopped,1/2 cup Walnuts - finely chopped,1 teaspoon Mustard seeds,2 teaspoon Turmeric 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
chef('To make mater paneer')

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'To make mater paneer, first wash and soak the mooli/grinder in water for 2 hours. Add the chopped cumin seeds and allow it to crackle. After 2 hours add the soaked mutton and red chillies. Let it crackle for 1-2 minutes till the mutton becomes soft. Let it cool. Once cooled, grind the ghee and ginger in a mixer grinder, add the chopped coriander leaves to it and keep it aside.Heat oil in a pan, add the mustard seeds and urad dal and let it splutter.Add the chopped coriander leaves and curry leaves and saute for a minute.Add the drained mooli / grinder and mix well.Add the grated coconut and saute for a minute. Add the chopped spinach, green chillies and cook it for a minute till it turns soft.Add the chopped raw mango, turmeric powder, coriander powder, dry red chillies, turmeric powder and salt. Mix well and cook for a minute. Add the mango and mix well, cover and cook till the raw mango becomes soft.Add the chopped green chillies and mix well.Add the chopped cori

In [ ]:
chef('To begin making Gongura Chicken')


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'To begin making Gongura Chicken Recipe first, heat ghee in a pan. Add the cumin seeds, sesame seeds and let them splutter.Add the garlic, curry leaves, green chilies and saute till it softens.Once the garlic has softened, add the ground spice paste, cumin powder and saute for a few seconds.Add the chicken and saute for a few seconds. Add the cooked rice, salt - as per taste and mix well.Check the salt and seasonings and adjust to suit your taste.Now, add the grated ginger, green chillies and cumin and saute for a few seconds.Add the cooked rice, mix thoroughly and let it cook for a few minutes. Add the chopped onion and saute for a couple of seconds.Add the red chilli powder, turmeric powder, cumin powder, chilli powder, red chili powder and saute for a few seconds.Add the onion mixture, turmeric powder, coriander powder, garam masala powder, salt and mix well.Add the cooked chicken, mix and cook for about 1 minute.Add the remaining ingredients to the chicken, mix 

In [ ]:
chef('4 Dry Red Chillies,1/2')[0]['generated_text']

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

# After successful login, you can push your model to the Hub
# Replace "my-awesome-indian-chef-model" with your desired model name on the Hub
trainer.push_to_hub("gpt2-indian-chef-model")